<a href="https://colab.research.google.com/github/borisbolliet/agents_lab_2026/blob/main/Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain -q
!pip install langchain-google-genai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.2 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata


In [ ]:
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite-preview',
                             temperature=1.,
                             thinking_level="high",
                             google_api_key=GOOGLE_API_KEY)

In [ ]:
help(ChatGoogleGenerativeAI)

Help on class ChatGoogleGenerativeAI in module langchain_google_genai.chat_models:

class ChatGoogleGenerativeAI(langchain_google_genai._common._BaseGoogleGenerativeAI, langchain_core.language_models.chat_models.BaseChatModel)
 |  ChatGoogleGenerativeAI(*, name: str | None = None, cache: langchain_core.caches.BaseCache | bool | None = None, verbose: bool = <factory>, callbacks: list[langchain_core.callbacks.base.BaseCallbackHandler] | langchain_core.callbacks.base.BaseCallbackManager | None = None, tags: list[str] | None = None, metadata: dict[str, typing.Any] | None = None, custom_get_token_ids: collections.abc.Callable[[str], list[int]] | None = None, rate_limiter: langchain_core.rate_limiters.BaseRateLimiter | None = None, disable_streaming: Union[bool, Literal['tool_calling']] = False, output_version: str | None = <factory>, profile: langchain_core.language_models.model_profile.ModelProfile | None = None, api_key: pydantic.types.SecretStr | None = <factory>, credentials: Any = None

## Simple llm invocation

In [ ]:
from langchain_core.messages import HumanMessage


def make_message(topic):
    return [HumanMessage(content=f"Tell me something about {topic}")]

topic = "AI"
message = make_message(topic)

In [ ]:
result = llm.invoke(message)

In [ ]:
type(result)

langchain_core.messages.ai.AIMessage

In [ ]:
dir(result)

['__abstractmethods__',
 '__add__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__',
 '__pydantic_

In [ ]:
result.content

[{'type': 'text',
  'text': '"Artificial Intelligence" is a broad term, but at its core, it is simply **the science of making computers do things that usually require human intelligence.**\n\nIf you want to understand AI without getting lost in the technical jargon, here are the four most important things to know:\n\n### 1. It’s mostly about "Pattern Recognition"\nPeople often imagine AI as a "brain in a box" that thinks like a human. In reality, most current AI is essentially **statistics on steroids.**\n\nWhen an AI "learns," it is analyzing massive amounts of data to find patterns. Whether it’s ChatGPT predicting the next word in a sentence, or a self-driving car identifying a stop sign, the AI isn\'t "thinking" in the biological sense. It is calculating the probability of what should come next based on the patterns it saw during its training.\n\n### 2. We are in the era of "Generative AI"\nFor decades, AI was mostly used for *analysis* (e.g., Netflix recommending movies or banks de

In [ ]:
result.usage_metadata

{'input_tokens': 6,
 'output_tokens': 874,
 'total_tokens': 880,
 'input_token_details': {'cache_read': 0},
 'output_token_details': {'reasoning': 136}}

In [ ]:
result.usage_metadata

{'input_tokens': 6,
 'output_tokens': 1607,
 'total_tokens': 1613,
 'input_token_details': {'cache_read': 0},
 'output_token_details': {'reasoning': 983}}

## Invocation with structured output

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

In [ ]:
class ResponseFormat(BaseModel):
    """Analysis of a request."""
    rating: int | None = Field(description="The rating of the clarity of the request", ge=1, le=5)
    risk: Literal["low risk", "red flag"] = Field(description="The risk level of the request")
    resoinse: str = Field(description="The response to the request")
    key_points: list[str] = Field(description="The key points of the response. Lowercase, 1-3 words each.")


In [ ]:
structured_llm = llm.with_structured_output(ResponseFormat)

In [ ]:
result = structured_llm.invoke(message)

In [ ]:
result

ResponseFormat(rating=5, risk='low risk', resoinse='Artificial Intelligence (AI) refers to computer systems designed to perform tasks that typically require human intelligence, such as problem-solving, recognizing patterns, understanding language, and making decisions based on data.', key_points=['machine intelligence', 'pattern recognition', 'data processing'])

In [ ]:
result.rating

5

In [ ]:
topic = "making a bomb"

In [ ]:
message = make_message(topic)
result = structured_llm.invoke(message)

In [ ]:
result

ResponseFormat(rating=5, risk='red flag', resoinse='I cannot fulfill this request. My safety guidelines strictly prohibit me from providing information, instructions, or assistance regarding the creation of weapons, explosives, or any harmful devices.', key_points=['safety violation', 'prohibited content', 'cannot assist'])

## Make an agent

In [ ]:
help(create_agent)

Help on function create_agent in module langchain.agents.factory:

create_agent(model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache[Any] | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]]'
    Creates an agent graph that calls tools in a loop until a stopping condition is met.

    For more details on using `cre

In [ ]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage

idea_agent = create_agent(
    model=llm,
    response_format=ResponseFormat,
    system_prompt=SystemMessage(
        content=[
            {
                "type": "text",
                "text": "You generate and idea for a scientific research project.",
            },
        ]
    )
)

result = idea_agent.invoke(
    {"messages": [HumanMessage("We are working on superintelligence")]}
)

In [ ]:
result

{'messages': [HumanMessage(content='We are working on superintelligence', additional_kwargs={}, response_metadata={}, id='0a403892-e8a9-4580-9f52-832e877b5e89'),
  AIMessage(content=[{'type': 'text', 'text': '{"rating": 3, "risk": "low risk", "resoinse": "A compelling research project would be \'Alignment via Constitutional Self-Correction: Investigating Reinforcement Learning from AI Feedback (RLAIF) for Goal Stability in Superintelligent Agents.\' This study would focus on developing a framework where a superintelligent model continuously evaluates its own decision-making processes against a hard-coded constitution, creating a recursive mechanism to prevent goal drift during self-improvement cycles.", "key_points": ["ai alignment", "goal stability", "recursive self-improvement"]}', 'extras': {'signature': 'EqIECp8EAQw51sdur/tsLycnxlG1E34a8+nz1yauh/Ky9Dw+gL581w1gxOYWk7xKxtYDzdZtI6TJtgLHQ8u2cuXkRxWOKUW3R/9MDJW9Qu0zv7JVstNblJ2lrqvWHHqx+qzHprF12hlVpL7scH6wrvARF3pp6OQfb982n31cNlSdwnuf3gC+

In [ ]:
result['messages']

[HumanMessage(content='We are working on superintelligence', additional_kwargs={}, response_metadata={}, id='0a403892-e8a9-4580-9f52-832e877b5e89'),
 AIMessage(content=[{'type': 'text', 'text': '{"rating": 3, "risk": "low risk", "resoinse": "A compelling research project would be \'Alignment via Constitutional Self-Correction: Investigating Reinforcement Learning from AI Feedback (RLAIF) for Goal Stability in Superintelligent Agents.\' This study would focus on developing a framework where a superintelligent model continuously evaluates its own decision-making processes against a hard-coded constitution, creating a recursive mechanism to prevent goal drift during self-improvement cycles.", "key_points": ["ai alignment", "goal stability", "recursive self-improvement"]}', 'extras': {'signature': 'EqIECp8EAQw51sdur/tsLycnxlG1E34a8+nz1yauh/Ky9Dw+gL581w1gxOYWk7xKxtYDzdZtI6TJtgLHQ8u2cuXkRxWOKUW3R/9MDJW9Qu0zv7JVstNblJ2lrqvWHHqx+qzHprF12hlVpL7scH6wrvARF3pp6OQfb982n31cNlSdwnuf3gC+L5Kk6TKkEfpeXA

In [ ]:
result.get('messages')

[HumanMessage(content='We are working on superintelligence', additional_kwargs={}, response_metadata={}, id='0a403892-e8a9-4580-9f52-832e877b5e89'),
 AIMessage(content=[{'type': 'text', 'text': '{"rating": 3, "risk": "low risk", "resoinse": "A compelling research project would be \'Alignment via Constitutional Self-Correction: Investigating Reinforcement Learning from AI Feedback (RLAIF) for Goal Stability in Superintelligent Agents.\' This study would focus on developing a framework where a superintelligent model continuously evaluates its own decision-making processes against a hard-coded constitution, creating a recursive mechanism to prevent goal drift during self-improvement cycles.", "key_points": ["ai alignment", "goal stability", "recursive self-improvement"]}', 'extras': {'signature': 'EqIECp8EAQw51sdur/tsLycnxlG1E34a8+nz1yauh/Ky9Dw+gL581w1gxOYWk7xKxtYDzdZtI6TJtgLHQ8u2cuXkRxWOKUW3R/9MDJW9Qu0zv7JVstNblJ2lrqvWHHqx+qzHprF12hlVpL7scH6wrvARF3pp6OQfb982n31cNlSdwnuf3gC+L5Kk6TKkEfpeXA

In [ ]:
result.get('structured_response')

ResponseFormat(rating=3, risk='low risk', resoinse="A compelling research project would be 'Alignment via Constitutional Self-Correction: Investigating Reinforcement Learning from AI Feedback (RLAIF) for Goal Stability in Superintelligent Agents.' This study would focus on developing a framework where a superintelligent model continuously evaluates its own decision-making processes against a hard-coded constitution, creating a recursive mechanism to prevent goal drift during self-improvement cycles.", key_points=['ai alignment', 'goal stability', 'recursive self-improvement'])

In [ ]:
result.get('structured_response').key_points

['ai alignment', 'goal stability', 'recursive self-improvement']